# Conversation Evaluation Benchmark

## Goal
Design a production-ready benchmark to score conversation turns on 300+ facets (linguistic quality, pragmatics, safety, emotion).

## Key Solutions
1. **Scalable Batching**: Logic to handle 300 (or 5000) facets by chunking them into smaller evaluation batches.
2. **Open Weights**: Optimized for model sizes <= 16B (Llama 3.1-8B).
3. **Confidence Scoring**: Each score is accompanied by a reasoning and a confidence score (1-5).
4. **Data Augmentation**: Categorization and type assignment for facets.

## 0. Setup & Credentials
Run this cell and enter your Groq API Key when prompted.

In [ ]:
# Install dependencies
!pip install -q pandas numpy litellm tqdm python-dotenv gradio

import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    print("Please enter your Groq API Key (from console.groq.com):")
    os.environ["GROQ_API_KEY"] = getpass()

## 1. Data Cleaning & Preprocessing

Clean the CSV, extract categories, and add metadata.

In [ ]:
import pandas as pd
import re

def clean_facet_name(text):
    text = re.sub(r'^\d+\.\s*', '', str(text))
    return text.strip().rstrip(':')

try:
    df = pd.read_csv('Facets Assignment.csv')
    df['Cleaned_Facet'] = df['Facets'].apply(clean_facet_name)
    df['Is_Category'] = df['Facets'].str.strip().str.endswith(':')
    
    current_cat = "General"
    categories = []
    for idx, row in df.iterrows():
        if row['Is_Category']:
            current_cat = row['Cleaned_Facet']
        categories.append(current_cat)
    df['Category'] = categories
    
    facets_df = df[~df['Is_Category']].copy()
    facets_df['Weight'] = 1.0
    facets_df['Evaluation_Metric'] = "Likert 1-5"
    
    print(f"Preprocessed {len(facets_df)} facets across {len(facets_df['Category'].unique())} categories.")
    display(facets_df.head())
except FileNotFoundError:
    print("Error: 'Facets Assignment.csv' not found. Please upload it to Colab.")

## 2. Evaluation Engine (Scalable Architecture)

This engine supports any number of facets by processing them in logical batches.

In [ ]:
import json
import litellm
from litellm import completion
import os

class EvaluationEngine:
    def __init__(self, model="groq/llama-3.1-8b-instant", batch_size=12):
        self.model = model
        self.batch_size = batch_size

    def _get_prompt(self, conv, facets):
        facet_text = "\n".join([f"- {f}" for f in facets])
        return f"""Analyze the turn below based on these specific facets.
Scale: 1 (Lowest/Worst) to 5 (Highest/Best).

History: {conv['history']}
Target Turn: {conv['target_turn']}

Facets:
{facet_text}

Respond in JSON format ONLY using this structure:
{{
  "results": [
    {{"facet": "name", "score": int, "reasoning": "string", "confidence": int}}
  ]
}}"""

    def evaluate(self, conversation, facets):
        all_results = []
        for i in range(0, len(facets), self.batch_size):
            batch = facets[i : i + self.batch_size]
            prompt = self._get_prompt(conversation, batch)
            
            try:
                response = completion(
                    model=self.model, 
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"}
                )
                content = response.choices[0].message.content
                batch_results = json.loads(content)
                res = batch_results.get('results', batch_results.get('evaluations', []))
                all_results.extend(res)
            except Exception as e:
                print(f"Error with batch starting at {i}: {e}")
        return all_results

## 3. Dataset Generation

Generating 50 synthetic conversations for benchmarking.

In [ ]:
import random
try:
    with open('sample_base.json', 'r') as f: base_samples = json.load(f)
except:
    base_samples = [{"history": "Hi", "target_turn": "Hello! How can I help?", "scenario": "General"}]

expanded_data = []
for i in range(50):
    base = random.choice(base_samples)
    expanded_data.append({
        "id": i + 1,
        "history": base['history'],
        "target_turn": f"{base['target_turn']} (Scenario Adaptation {i+1})",
        "scenario": base['scenario']
    })

print(f"Created {len(expanded_data)} test cases.")

## 4. Run Benchmark

Process the dataset. (Starting with a small subset for verification).

In [ ]:
engine = EvaluationEngine()
test_facets = facets_df['Cleaned_Facet'].tolist()[:15] # Subset for initial testing

results_log = []
to_process = expanded_data[:3] # Set to expanded_data for full run

for conv in to_process:
    print(f"Scoring conversation {conv['id']}...")
    scores = engine.evaluate(conv, test_facets)
    results_log.append({"conversation": conv, "scores": scores})

with open('final_evaluation.json', 'w') as f:
    json.dump(results_log, f, indent=2)

print("\nBenchmark complete. Results saved to 'final_evaluation.json'.")

## 5. View Results Dashboard

Visualizing the results.

In [ ]:
import gradio as gr

def view_scores(conv_index):
    idx = int(conv_index) - 1
    if 0 <= idx < len(results_log):
        df_res = pd.DataFrame(results_log[idx]['scores'])
        return df_res
    return pd.DataFrame([{"Error": "Conversation index out of range"}])

if results_log:
    interface = gr.Interface(
        fn=view_scores, 
        inputs=gr.Number(label="Conversation (1-based index)", value=1),
        outputs="dataframe",
        title="Facet Evaluation Result Viewer"
    )
    interface.launch(share=True)
else:
    print("No results to display. Please run the benchmark cell first.")